# 이미지 압축 비교 (교과서 57쪽)

같은 그림을 **BMP, PNG, GIF**로 저장해 해상도와 파일 크기를 비교해요.

- **PNG**: 그림의 정보를 변형하지 않고 용량만 줄이는 무손실 압축 방법
- **GIF**: 무손실 압축 방법이지만 256색만 표현할 수 있어 그림의 품질은 낮고, 애니메이션 기능이 있음

위에서부터 셀을 하나씩 실행해요(셀 왼쪽의 ▶ 버튼 또는 Shift + Enter).

## 1. 이미지 변환에 필요한 라이브러리 불러오기
**PIL**(Python Imaging Library)은 이미지 분석 및 처리를 쉽게 할 수 있는 라이브러리예요.

In [ ]:
import os
from PIL import Image, ImageDraw
import random

src = None   # 내 사진 파일 이름(사진을 올리지 않으면 None 그대로)

## 2. (선택) 내 사진 올리기
내 사진으로 해 보고 싶으면 아래 셀을 실행하고 **파일 선택**을 눌러 사진 한 장을 골라요.
사람 얼굴이나 개인 정보가 드러난 사진은 쓰지 않아요. **건너뛰면 코드로 만든 그림**을 써요.

In [ ]:
from google.colab import files
up = files.upload()
if up:
    src = list(up.keys())[0]
    print('올린 파일:', src)

## 3. 원본 BMP 이미지 만들기
올린 사진(없으면 코드로 그린 꽃 그림)을 압축하지 않은 **BMP** 파일로 저장해 원본으로 써요.

In [ ]:
def make_picture(w=480, h=320, seed=57):
    # 하늘색 배경에 꽃잎이 흩어진 그림을 코드로 그려요.
    rnd = random.Random(seed)
    im = Image.new('RGB', (w, h))
    px = im.load()
    for y in range(h):
        for x in range(w):
            n = rnd.randint(-12, 12)
            px[x, y] = (47 + y // 4 + n, 111 + y // 3 + n, 196 + y // 6 + n)
    d = ImageDraw.Draw(im)
    for i in range(400):
        x, y, r = rnd.randint(0, w), rnd.randint(20, h - 20), rnd.randint(3, 11)
        d.ellipse((x - r, y - r, x + r, y + r), fill=(255, rnd.randint(170, 225), rnd.randint(200, 235)))
    return im

if src:
    im0 = Image.open(src).convert('RGB')
    im0.thumbnail((640, 640))
else:
    im0 = make_picture()
im0.save('my_BMP.bmp', 'bmp')
print('원본 BMP를 만들었어요.')

## 4. BMP 이미지의 해상도와 파일 크기 확인하기

In [ ]:
im = Image.open('my_BMP.bmp')
print('이미지 확장자', im.format)
print('이미지 해상도', im.size)
print('파일 크기', os.path.getsize('my_BMP.bmp'), 'Bytes')

## 5. 같은 이미지를 PNG, GIF로 변환해 해상도와 파일 크기 확인하기

In [ ]:
im.save('my_PNG.png', 'png')
im2 = Image.open('my_PNG.png')
print('이미지 확장자', im2.format)
print('이미지 해상도', im2.size)
print('파일 크기', os.path.getsize('my_PNG.png'), 'Bytes')

im.save('my_GIF.gif', 'gif')
im3 = Image.open('my_GIF.gif')
print('이미지 확장자', im3.format)
print('이미지 해상도', im3.size)
print('파일 크기', os.path.getsize('my_GIF.gif'), 'Bytes')

## 6. 압축률 비교하기
압축률 = 변환하기 전 크기 ÷ 변환한 뒤 크기 (높을수록 압축이 더 잘된 것)

In [ ]:
bmp = os.path.getsize('my_BMP.bmp')
for name in ['my_BMP.bmp', 'my_PNG.png', 'my_GIF.gif']:
    size = os.path.getsize(name)
    print(name, size, 'Bytes', '압축률', round(bmp / size, 2))

## 7. BMP, PNG, GIF 이미지를 출력해 눈으로 차이 구분하기
`display()` 명령어에 이미지 변수를 넣으면 코랩이 그림을 셀 아래에 보여 줘요. 위에서부터 차례대로 BMP(`im`), PNG(`im2`), GIF(`im3`)예요.

In [ ]:
display(im)
display(im2)
display(im3)

### 확대해서 비교하기
가운데 부분을 크게 키워 보면 GIF의 색이 줄어든 것이 더 잘 보여요. 왼쪽부터 BMP, PNG, GIF예요.

In [ ]:
w, h = im.size
box = (w // 2 - 30, h // 2 - 20, w // 2 + 30, h // 2 + 20)
parts = [Image.open(f).convert('RGB').crop(box).resize((240, 160), Image.NEAREST) for f in ['my_BMP.bmp', 'my_PNG.png', 'my_GIF.gif']]
board = Image.new('RGB', (740, 160), 'white')
for i, p in enumerate(parts):
    board.paste(p, (i * 250, 0))
board

## 8. 이야기 나누기
세 가지 압축 방법을 비교해 보고, 어떤 압축 방법이 가장 효율적일지 품질과 용량을 근거로 이야기 나누어 보세요.

## (더 해 보기) JPEG와도 비교하기
JPEG는 손실 압축 방법이에요. `quality` 값을 바꿔 가며 파일 크기를 비교해 보세요.

In [ ]:
for q in [90, 50, 10]:
    im.save(f'my_JPEG_{q}.jpg', 'jpeg', quality=q)
    size = os.path.getsize(f'my_JPEG_{q}.jpg')
    print('품질', q, '파일 크기', size, 'Bytes', '압축률', round(bmp / size, 2))
display(Image.open('my_JPEG_10.jpg'))